# run76 — strengthening the fNIRS decision path

**Why:** in run69 the fNIRS branch alone reached **0.6167**, above the model's own final output (0.6037). The fNIRS path is this model's strongest single predictor, so it is the thing worth strengthening. Two independent experiments here, no new preprocessing.

Baseline seed 42: window **0.6074**, trial-level **0.6325**.

## Part A — `--lambda-fnirs` (no new code; the flag already existed)

`l_class = l_pred + 0.2·l_eeg + 0.2·l_fnirs + l_fusion`: the fNIRS branch's own loss carries 5× less weight than the main paths. MGFNet gives its auxiliary heads more total weight than its main head. Run A1 first.

In [ ]:
%run run_all_subjects.py --seed 42 --adabn --fnirs-temporal-align egta --efgf-gate-mode sigmoid --efgf-gate-init -2.0 --normalize-ypred --no-decay-scalars --stage2-stop-mode none --lambda-fnirs 0.5

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
%run run_all_subjects.py --seed 42 --adabn --fnirs-temporal-align egta --efgf-gate-mode sigmoid --efgf-gate-init -2.0 --normalize-ypred --no-decay-scalars --stage2-stop-mode none --lambda-fnirs 1.0

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

## Part B — `--fnirs-chroma-split` (priority-CHROMASPLIT, MGFNet)

HbO and HbR get their own `FNIRSBranch` with non-shared weights, added to the fNIRS **decision** path (`fn_pooled` → CAFE/EGTA → `clf_fnirs`) through zero-init 1×1 convs. EFGF and the fusion branch are untouched, so only the fNIRS path changes. `separate` adds ~17.8k parameters, `separate_pair` ~28.2k.

Verified on CPU: at initialisation the output equals the baseline exactly; the new branches start frozen and begin learning from step 1 (standard for this zero-init pattern, same as `--fusion-fnirs-inject`).

In [ ]:
%run run_all_subjects.py --seed 42 --adabn --fnirs-temporal-align egta --efgf-gate-mode sigmoid --efgf-gate-init -2.0 --normalize-ypred --no-decay-scalars --stage2-stop-mode none --fnirs-chroma-split separate

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

### only if `separate` improved

In [ ]:
%run run_all_subjects.py --seed 42 --adabn --fnirs-temporal-align egta --efgf-gate-mode sigmoid --efgf-gate-init -2.0 --normalize-ypred --no-decay-scalars --stage2-stop-mode none --fnirs-chroma-split separate_pair

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

## trial-level evaluation — pass the SAME flags as the run

`--lambda-fnirs` is a training-only flag, so trial_level_eval needs no extra argument for Part A. `--fnirs-chroma-split` changes the architecture, so it MUST be repeated.

In [ ]:
%run trial_level_eval.py --seed 42 --variant-hash <hash> --expected-window-acc <grand_42>

In [ ]:
%run trial_level_eval.py --seed 42 --variant-hash <hash> --expected-window-acc <grand_42> --fnirs-chroma-split separate

## seeds 999 / 2024 — only for a winner

In [ ]:
%run run_all_subjects.py --seed 999 --adabn --fnirs-temporal-align egta --efgf-gate-mode sigmoid --efgf-gate-init -2.0 --normalize-ypred --no-decay-scalars --stage2-stop-mode none <winning flag>

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
%run run_all_subjects.py --seed 2024 --adabn --fnirs-temporal-align egta --efgf-gate-mode sigmoid --efgf-gate-init -2.0 --normalize-ypred --no-decay-scalars --stage2-stop-mode none <winning flag>